In [1]:
import sys
sys.path.append("../src")

In [2]:
import time
import httpx
import pandas as pd

In [3]:
from smogcast.ingest.gios import get_all_stations

In [4]:
stations = get_all_stations()

In [5]:
len(stations)

288

In [6]:
stations_with_pm_poland = []

for i, station in enumerate(stations, start=1):
    print(f"{i}/{len(stations)} - {station['Nazwa stacji']}")

    station_id = station["Identyfikator stacji"]

    url = (
        "https://api.gios.gov.pl/pjp-api/"
        f"v1/rest/station/sensors/{station_id}"
    )

    response = httpx.get(url, timeout=30.0)
    response.raise_for_status()

    sensor_data = response.json()

    sensors = sensor_data[
        "Lista stanowisk pomiarowych dla podanej stacji"
    ]

    pm_sensors = [
        sensor
        for sensor in sensors
        if sensor.get("Wskaźnik") in {
            "pył zawieszony PM10",
            "pył zawieszony PM2.5",
        }
    ]

    if pm_sensors:
        stations_with_pm_poland.append(
            {
                "station_id": station_id,
                "station_name": station["Nazwa stacji"],
                "city": station["Nazwa miasta"],
                "voivodeship": station["Województwo"],
                "pm_sensors": pm_sensors,
            }
        )

    time.sleep(0.2)

1/288 - Czerniawa
2/288 - Dzierżoniów, ul. Piłsudskiego
3/288 - Kłodzko, ul. Szkolna
4/288 - Legnica, Al. Rzeczypospolitej
5/288 - Oława, ul. Żołnierzy Armii Krajowej
6/288 - Oleśnica, ul. Brzozowa
7/288 - Osieczów
8/288 - Polkowice, ul. Kasztanowa
9/288 - Szczawno-Zdrój, ul. Kolejowa
10/288 - Wałbrzych, ul. Wysockiego
11/288 - Wrocław, ul. Bartnicza
12/288 - Wrocław, ul. Na Grobli
13/288 - Wrocław, wyb. Conrada-Korzeniowskiego
14/288 - Wrocław, ul. Orzechowa
15/288 - Wrocław, Al. Wiśniowa
16/288 - Zgorzelec, ul. Bohaterów Getta
17/288 - Grudziądz, ul. Piłsudskiego
18/288 - Inowrocław, ul. Solankowa
19/288 - Toruń, ul. Wały gen. Sikorskiego
20/288 - Bydgoszcz, ul. Fieldorfa
21/288 - Bydgoszcz, Pl. Poznański
22/288 - Bydgoszcz, ul. Warszawska
23/288 - Ciechocinek, ul. Tężniowa
24/288 - Grudziądz, ul. Sienkiewicza
25/288 - Koniczynka, Pojezierze Chełmińskie
26/288 - Toruń, ul. Dziewulskiego
27/288 - Toruń, ul. Przy Kaszowniku
28/288 - Włocławek, ul. Okrzei
29/288 - Zielonka, Bory Tuchols

In [7]:
stations_pm_poland_df = pd.DataFrame(
    [
        {
            "station_id": station["station_id"],
            "station_name": station["station_name"],
            "city": station["city"],
            "voivodeship": station["voivodeship"],
            "pm10_count": sum(
                sensor.get("Wskaźnik") == "pył zawieszony PM10"
                for sensor in station["pm_sensors"]
            ),
            "pm25_count": sum(
                sensor.get("Wskaźnik") == "pył zawieszony PM2.5"
                for sensor in station["pm_sensors"]
            ),
        }
        for station in stations_with_pm_poland
    ]
)

In [8]:
stations_pm_poland_df

,station_id,station_name,city,voivodeship,pm10_count,pm25_count
0,11,Czerniawa,Czerniawa,DOLNOŚLĄSKIE,1,0
1,16,"Dzierżoniów, ul. Piłsudskiego",Dzierżoniów,DOLNOŚLĄSKIE,1,0
2,38,"Kłodzko, ul. Szkolna",Kłodzko,DOLNOŚLĄSKIE,1,1
3,52,"Legnica, Al. Rzeczypospolitej",Legnica,DOLNOŚLĄSKIE,2,1
4,70,"Oława, ul. Żołnierzy Armii Krajowej",Oława,DOLNOŚLĄSKIE,2,1
...,...,...,...,...,...,...
260,20568,"Solec-Zdrój, ul. Leśna",Solec Zdrój,ŚWIĘTOKRZYSKIE,2,1
261,20590,ul. Sportowa 15,Kobierzyce,DOLNOŚLĄSKIE,1,0
262,20591,"Solec Kujawski, ul. Tartaczna 10",Solec Kujawski,KUJAWSKO-POMORSKIE,1,1
263,20607,"Tczew, ul. Czyżykowska",Tczew,POMORSKIE,1,1


In [9]:
len(stations_pm_poland_df)

265

In [10]:
stations_both_df = stations_pm_poland_df[
    (stations_pm_poland_df["pm10_count"] > 0)
    & (stations_pm_poland_df["pm25_count"] > 0)
].copy()

In [11]:
stations_both_df

,station_id,station_name,city,voivodeship,pm10_count,pm25_count
2,38,"Kłodzko, ul. Szkolna",Kłodzko,DOLNOŚLĄSKIE,1,1
3,52,"Legnica, Al. Rzeczypospolitej",Legnica,DOLNOŚLĄSKIE,2,1
4,70,"Oława, ul. Żołnierzy Armii Krajowej",Oława,DOLNOŚLĄSKIE,2,1
8,109,"Wałbrzych, ul. Wysockiego",Wałbrzych,DOLNOŚLĄSKIE,2,2
10,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,1,1
...,...,...,...,...,...,...
259,20567,"Ostrowiec Świętokrzyski, ul. Polna",Ostrowiec Świętokrzyski,ŚWIĘTOKRZYSKIE,2,1
260,20568,"Solec-Zdrój, ul. Leśna",Solec Zdrój,ŚWIĘTOKRZYSKIE,2,1
262,20591,"Solec Kujawski, ul. Tartaczna 10",Solec Kujawski,KUJAWSKO-POMORSKIE,1,1
263,20607,"Tczew, ul. Czyżykowska",Tczew,POMORSKIE,1,1


In [12]:
len(stations_both_df)

144

In [13]:
stations_both_df["voivodeship"].value_counts()

voivodeship
MAZOWIECKIE            20
PODKARPACKIE           13
ŚLĄSKIE                13
DOLNOŚLĄSKIE           11
MAŁOPOLSKIE            10
KUJAWSKO-POMORSKIE      9
PODLASKIE               9
POMORSKIE               9
LUBUSKIE                8
ŚWIĘTOKRZYSKIE          8
LUBELSKIE               7
WIELKOPOLSKIE           7
ZACHODNIOPOMORSKIE      6
ŁÓDZKIE                 5
OPOLSKIE                5
WARMIŃSKO-MAZURSKIE     4
Name: count, dtype: int64

In [14]:
capital_cities = {
    "DOLNOŚLĄSKIE": "Wrocław",
    "KUJAWSKO-POMORSKIE": "Bydgoszcz",
    "LUBELSKIE": "Lublin",
    "LUBUSKIE": "Gorzów Wielkopolski",
    "ŁÓDZKIE": "Łódź",
    "MAŁOPOLSKIE": "Kraków",
    "MAZOWIECKIE": "Warszawa",
    "OPOLSKIE": "Opole",
    "PODKARPACKIE": "Rzeszów",
    "PODLASKIE": "Białystok",
    "POMORSKIE": "Gdańsk",
    "ŚLĄSKIE": "Katowice",
    "ŚWIĘTOKRZYSKIE": "Kielce",
    "WARMIŃSKO-MAZURSKIE": "Elbląg",
    "WIELKOPOLSKIE": "Poznań",
    "ZACHODNIOPOMORSKIE": "Szczecin",
}

In [15]:
capital_stations = []

for voivodeship, city in capital_cities.items():
    matches = stations_both_df[
        (stations_both_df["voivodeship"] == voivodeship)
        & (stations_both_df["city"] == city)
    ]

    if not matches.empty:
        capital_stations.append(matches.iloc[0])
    else:
        print(f"Brak pasującej stacji: {voivodeship} - {city}")

In [16]:
capital_stations_df = pd.DataFrame(
    capital_stations
).reset_index(drop=True)

In [17]:
capital_stations_df[
    [
        "voivodeship",
        "city",
        "station_name",
        "pm10_count",
        "pm25_count",
    ]
]

,voivodeship,city,station_name,pm10_count,pm25_count
0,DOLNOŚLĄSKIE,Wrocław,"Wrocław, wyb. Conrada-Korzeniowskiego",1,1
1,KUJAWSKO-POMORSKIE,Bydgoszcz,"Bydgoszcz, Pl. Poznański",1,1
2,LUBELSKIE,Lublin,"Lublin, ul. Śliwińskiego",1,1
3,LUBUSKIE,Gorzów Wielkopolski,Gorzów Wlkp. ul. Kosynierów Gdyńskich,2,1
4,ŁÓDZKIE,Łódź,"Łódź, ul. Czernika",1,2
5,MAŁOPOLSKIE,Kraków,"Kraków, Aleja Krasińskiego",1,1
6,MAZOWIECKIE,Warszawa,"Warszawa, Al. Niepodległości",1,1
7,OPOLSKIE,Opole,"Opole, Os. Armii Krajowej",1,1
8,PODKARPACKIE,Rzeszów,"Rzeszów, Al. Rejtana",2,1
9,PODLASKIE,Białystok,"Białystok, ul. Warszawska",1,2


In [18]:
len(capital_stations_df)

16

In [19]:
(
    (capital_stations_df["pm10_count"] > 0)
    & (capital_stations_df["pm25_count"] > 0)
).all()

np.True_

In [20]:
capital_station_ids = capital_stations_df[
    "station_id"
].tolist()

In [21]:
capital_station_ids

[117,
 156,
 258,
 361,
 295,
 400,
 530,
 590,
 671,
 609,
 729,
 814,
 11195,
 861,
 944,
 986]

In [22]:
len(capital_station_ids)

16

In [23]:
sensor_records = []

for station in stations_with_pm_poland:
    if station["station_id"] not in capital_station_ids:
        continue

    for sensor in station["pm_sensors"]:
        indicator = sensor.get("Wskaźnik")

        if indicator == "pył zawieszony PM10":
            parameter = "PM10"

        elif indicator == "pył zawieszony PM2.5":
            parameter = "PM2.5"

        else:
            continue

        sensor_records.append(
            {
                "station_id": station["station_id"],
                "station_name": station["station_name"],
                "city": station["city"],
                "voivodeship": station["voivodeship"],
                "sensor_id": sensor["Identyfikator stanowiska"],
                "parameter": parameter,
            }
        )

In [24]:
sensor_records_df = pd.DataFrame(sensor_records)

In [25]:
sensor_records_df = (
    sensor_records_df
    .drop_duplicates(subset="sensor_id")
    .reset_index(drop=True)
)

In [26]:
sensor_records_df

,station_id,station_name,city,voivodeship,sensor_id,parameter
0,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,669,PM10
1,117,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,670,PM2.5
2,156,"Bydgoszcz, Pl. Poznański",Bydgoszcz,KUJAWSKO-POMORSKIE,965,PM10
3,156,"Bydgoszcz, Pl. Poznański",Bydgoszcz,KUJAWSKO-POMORSKIE,966,PM2.5
4,258,"Lublin, ul. Śliwińskiego",Lublin,LUBELSKIE,1832,PM10
5,258,"Lublin, ul. Śliwińskiego",Lublin,LUBELSKIE,1833,PM2.5
6,295,"Łódź, ul. Czernika",Łódź,ŁÓDZKIE,2069,PM10
7,295,"Łódź, ul. Czernika",Łódź,ŁÓDZKIE,2071,PM2.5
8,295,"Łódź, ul. Czernika",Łódź,ŁÓDZKIE,2072,PM2.5
9,361,Gorzów Wlkp. ul. Kosynierów Gdyńskich,Gorzów Wielkopolski,LUBUSKIE,2421,PM10


In [27]:
sensor_records_df["sensor_id"].duplicated().sum()

np.int64(0)

In [28]:
sensor_records_df.to_parquet(
    "../data/raw/gios_capital_sensors_poland.parquet",
    index=False,
)

In [29]:
sensor_records = sensor_records_df.to_dict("records")

In [30]:
len(sensor_records)

43

In [31]:
date_from = "2025-08-20 00:00"
date_to = "2026-08-20 23:00"

In [32]:
from smogcast.ingest.gios import get_data_for_sensor_records

In [33]:
all_measurements, failed_ranges = get_data_for_sensor_records(
    sensor_records,
    date_from,
    date_to,
    days_per_range=30,
)


1/43 | DOLNOŚLĄSKIE | Wrocław | PM10 | sensor 669
Pobieram sensor 669: 2025-08-20 00:00 - 2025-09-19 00:00
Pobieram sensor 669: 2025-09-19 00:00 - 2025-10-19 00:00
Pobieram sensor 669: 2025-10-19 00:00 - 2025-11-18 00:00
Pobieram sensor 669: 2025-11-18 00:00 - 2025-12-18 00:00
Pobieram sensor 669: 2025-12-18 00:00 - 2026-01-17 00:00
Pobieram sensor 669: 2026-01-17 00:00 - 2026-02-16 00:00
Pobieram sensor 669: 2026-02-16 00:00 - 2026-03-18 00:00
Pobieram sensor 669: 2026-03-18 00:00 - 2026-04-17 00:00
Pobieram sensor 669: 2026-04-17 00:00 - 2026-05-17 00:00
Pobieram sensor 669: 2026-05-17 00:00 - 2026-06-16 00:00
Pobieram sensor 669: 2026-06-16 00:00 - 2026-07-16 00:00
Pobieram sensor 669: 2026-07-16 00:00 - 2026-08-15 00:00
Pobieram sensor 669: 2026-08-15 00:00 - 2026-08-20 23:00

2/43 | DOLNOŚLĄSKIE | Wrocław | PM2.5 | sensor 670
Pobieram sensor 670: 2025-08-20 00:00 - 2025-09-19 00:00
Pobieram sensor 670: 2025-09-19 00:00 - 2025-10-19 00:00
Pobieram sensor 670: 2025-10-19 00:00 - 20

In [34]:
len(all_measurements)

225518

In [35]:
len(failed_ranges)

0

In [37]:
df = pd.DataFrame(all_measurements)

In [38]:
df.head()

,Nazwa stacji,Kod stanowiska,Data,Wartość,station_id,sensor_id,station_name,city,voivodeship,Parametr
0,"Wrocław, wyb. Conrada-Korzeniowskiego",DsWrocWybCon-PM10-24g,2025-08-20 00:00:00,29.9,117,669,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,PM10
1,"Wrocław, wyb. Conrada-Korzeniowskiego",DsWrocWybCon-PM10-24g,2025-08-21 00:00:00,23.0,117,669,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,PM10
2,"Wrocław, wyb. Conrada-Korzeniowskiego",DsWrocWybCon-PM10-24g,2025-08-22 00:00:00,21.2,117,669,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,PM10
3,"Wrocław, wyb. Conrada-Korzeniowskiego",DsWrocWybCon-PM10-24g,2025-08-23 00:00:00,23.5,117,669,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,PM10
4,"Wrocław, wyb. Conrada-Korzeniowskiego",DsWrocWybCon-PM10-24g,2025-08-24 00:00:00,10.2,117,669,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,PM10


In [39]:
df["Data"] = pd.to_datetime(df["Data"])

In [40]:
df = df[
    [
        "station_id",
        "sensor_id",
        "station_name",
        "city",
        "voivodeship",
        "Parametr",
        "Data",
        "Wartość",
        "Kod stanowiska",
    ]
]

In [41]:
print("Liczba pomiarów:", len(df))
print("Liczba stacji:", df["station_id"].nunique())
print("Liczba sensorów:", df["sensor_id"].nunique())
print("Najstarszy pomiar:", df["Data"].min())
print("Najnowszy pomiar:", df["Data"].max())

Liczba pomiarów: 225518
Liczba stacji: 16
Liczba sensorów: 43
Najstarszy pomiar: 2025-08-20 00:00:00
Najnowszy pomiar: 2026-08-20 23:00:00


In [42]:
df["Parametr"].value_counts()

Parametr
PM10     114250
PM2.5    111268
Name: count, dtype: int64

In [43]:
df.duplicated(
    subset=["sensor_id", "Data"]
).sum()

np.int64(504)

In [45]:
df = (
    df
    .drop_duplicates(
        subset=["sensor_id", "Data"],
        keep="first",
    )
    .reset_index(drop=True)
)

In [46]:
df.duplicated(
    subset=["sensor_id", "Data"]
).sum()

np.int64(0)

In [47]:
len(df)

225014

In [48]:
df.to_parquet(
    "../data/raw/gios_pm_measurements_1y.parquet",
    index=False,
)

In [49]:
failed_ranges_df = pd.DataFrame(
    failed_ranges,
    columns=["sensor_id", "date_from", "date_to"],
)

failed_ranges_df.to_csv(
    "../data/raw/gios_failed_ranges_1y.csv",
    index=False,
)